# A6 · LightGBM: let the computer choose the boxes

**Where we are**

```
0 Frame    → predict a SAFE slot (the q-point); q is the PO's setting (0.67 / 0.75 / 0.80)
1 Collect  → simulator made the visits
2 Explore  → found the problems
3 Prepare  → fixed them; split: train Jan–May, valid June, test Jul–Aug; built the 11 feature columns
4 Baseline → best simple method B3: 1.349 on June (q = 0.8); best possible 1.130
5 Train    → HERE: boosting, first by hand, then LightGBM
```

**Words for this lesson**

| Word | Plain meaning |
|---|---|
| tree | a chain of yes/no questions ("is it a follow-up?", "reports ≤ 3?") that sorts visits into boxes |
| leaf | the end of a branch: one box, with one number |
| round | one new small tree added to the model |
| push (gradient) | for each visit, the direction that lowers the score: up by q if it ran late, down by 1 − q if the doctor sat idle |
| learning rate | how big a step each tree takes (0.05 = 5% of the way) |
| overfitting | the model starts memorising individual training visits; the training score keeps improving while June gets worse |
| early stopping | stop adding trees once June stops improving, and keep the best number |

**Before running this notebook,** write the `TODO(you)` parts and make the tests pass:

| File | Your parts | Test |
|---|---|---|
| `slotpred/mini_boost.py` | `pinball_push`, `MiniBoost.fit` | `.venv/Scripts/python.exe -m pytest tests/test_mini_boost.py -v` |
| `slotpred/train_lgbm.py` | `make_model`, `fit_one` | `.venv/Scripts/python.exe -m pytest tests/test_train_lgbm.py -v` |

Everything is fitted on **train** and scored on **June**. The test months stay locked until A8.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from slotpred.config import OUT, Q_CANDIDATES
from slotpred.features import FEATURES, make_xy
from slotpred.metrics import evaluate, pinball_loss
from slotpred.mini_boost import MiniBoost, to_numeric
from slotpred.prepare import load_prepared
from slotpred.train_lgbm import PARAMS, fit_all, fit_one

splits, state = load_prepared()
X_train, y_train = make_xy(splits["train"], state)     # the 11 booking-time columns from A4
X_valid, y_valid = make_xy(splits["valid"], state)
baselines = pd.read_csv(OUT / "baselines_valid.csv")   # your A5 leaderboard
print(X_train.shape, X_valid.shape)
print(FEATURES)

## Part 1 · Boosting by hand (your `MiniBoost`)

q = 0.8, 100 rounds, each tree at most 8 boxes of at least 30 visits, learning rate 0.1. sklearn's trees need numbers, so `to_numeric` turns category columns into codes.

In [ ]:
q = 0.8
Xn_train, Xn_valid = to_numeric(X_train), to_numeric(X_valid)
mini = MiniBoost(q, n_rounds=100, learning_rate=0.1, max_leaf_nodes=8, min_samples_leaf=30).fit(Xn_train, y_train)
print("start (the training P80, like B1):", round(mini.start_, 2))
print("training score after rounds 1, 10, 50, 100:", [round(mini.history_[i], 3) for i in (0, 9, 49, 99)])

In [ ]:
# TODO(you): the June score after 1, 5, 10, 25, 50 and 100 rounds.
#   hint: pinball_loss(y_valid, mini.predict(Xn_valid, n_rounds=n), q)
# Then plot the training history (mini.history_) and your June scores against the number of rounds,
# with horizontal lines for B3 (1.349) and the best possible score (1.130).

**Questions**
1. Where does round 0 start, and which A5 baseline is that the same as?
2. After how many rounds does your hand-made booster beat B3 on June?
3. How close does it get to the best possible score?

*Answers:*

## A look inside one tree

Your first round's tree, as readable rules. Category columns appear as codes; `state.categories` (A4) lists them in order, so code 0 of `visit_type` is the first entry.

In [ ]:
from sklearn.tree import export_text
first_tree, first_values = mini.rounds_[0]
print(export_text(first_tree, feature_names=FEATURES, decimals=1))
print("visit_type codes:", dict(enumerate(state.categories["visit_type"])))
print("each box's step (q-point of its errors):", {k: round(v, 2) for k, v in first_values.items()})

**Question:** what is the first question the tree asks, and why do you think it chose that one first?

*Answer:*

## Part 2 · The real LightGBM (your `make_model` and `fit_one`)

Same idea, much faster and smarter: up to 2,000 trees of up to 31 boxes, learning rate 0.05, and early stopping watching June.

In [ ]:
model = fit_one(0.8, X_train, y_train, X_valid, y_valid)
print("trees kept:", model.best_iteration_)
print("June score:", round(pinball_loss(y_valid, model.predict(X_valid), 0.8), 3))

In [ ]:
# The training curve: score after every tree, on train and on June
hist = model.evals_result_
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(hist["train"]["quantile"], label="train (Jan–May)")
ax.plot(hist["valid"]["quantile"], label="June")
ax.axvline(model.best_iteration_, color="grey", ls="--", label=f"best: {model.best_iteration_} trees")
ax.axhline(1.130, color="red", ls=":", label="best possible (1.130)")
ax.set_xlabel("number of trees"); ax.set_ylabel("score (lower is better)"); ax.set_ylim(1.0, 1.6); ax.legend()
plt.show()

**Questions**
1. After the best point, what happens to the training curve, and what happens to June? What is that called?
2. Why do we keep the number of trees where **June** was best, not where training was best?

*Answers:*

## Part 3 · One model per q, against your baselines

`fit_all` trains one model for each q and saves them to `outputs/models/`.

In [ ]:
models = fit_all(X_train, y_train, X_valid, y_valid)

In [ ]:
# TODO(you): build the June leaderboard rows for LightGBM, one per q:
#   evaluate(y_valid, models[q].predict(X_valid), q, "LightGBM")
# Then put them together with your A5 baselines:  board = pd.concat([baselines, pd.DataFrame(rows)])
# Show method, q, pinball, coverage, booked_on_time, booked_idle_min, sorted by q then pinball.
# Finally save the LightGBM rows only:  pd.DataFrame(rows).to_csv(OUT / "lgbm_valid.csv", index=False)

**Questions**
1. For each q: how much of the gap between B3 and the best possible score does LightGBM close? (gap closed = (B3 − LightGBM) ÷ (B3 − best possible))
2. Is LightGBM's coverage close to q? Why does that matter (A1: coverage alone can be gamed, but it must still be right)?

*Answers:*

## Part 4 · Follow one visit through the trees

In [ ]:
m = models[0.8]
i = int(np.flatnonzero((splits["valid"]["department"] == "Neurology").to_numpy()
                       & (splits["valid"]["visit_type"] == "new").to_numpy())[0])
row = X_valid.iloc[[i]]
display(row.T)
print("actual minutes:", round(float(y_valid.iloc[i]), 1))
for k in (1, 2, 3, 10, 50, 100, m.best_iteration_):
    print(f"after {k:>3} trees: P80 = {m.predict(row, num_iteration=k)[0]:.2f}")

**Question:** the actual visit took less than the final P80. Is that a mistake? *(Hint: what does P80 promise about a single visit?)*

*Answer:*

## Part 5 · What did LightGBM rely on?

"Gain" = how much each column's questions lowered the score, added up over all trees, as a share of the total.

In [ ]:
gain = pd.Series(m.booster_.feature_importance("gain"), index=m.booster_.feature_name())
(gain / gain.sum()).sort_values().plot.barh(figsize=(7, 4), title="Share of total gain (q = 0.8)")
plt.show()

**Question:** compare the top columns with how we built the simulator in A2 (`world.py`). Does LightGBM's ranking make sense? Which column ranks low, and why might that be?

*Answer:*

## Wrap-up (answer in `notes/A06-lightgbm.md` too, in your own words)

1. `min_child_samples = 30` is the same idea as which A5 rule?
2. Learning rate 0.05: a tree thinks a visit needs +10 minutes. How much does that tree actually move it?
3. Why stop adding trees when June stops improving?
4. In one or two sentences: how does boosting turn many rough trees into one good prediction?